# Synthetic Data Generation Comparison: Faker vs SDV vs DataSynthesizer

This notebook takes a small input file containing **column names and allowed example values**, generates synthetic rows using three approaches, and then checks whether each generated output stayed within the provided examples.

## Input format

Create either `input_schema.csv` or `input_schema.xlsx` with these columns:

| column_name | examples |
|---|---|
| city | Delhi, Mumbai, Pune |
| status | ACTIVE, INACTIVE |
| amount | 100, 200, 500 |

The notebook treats every value in `examples` as an allowed domain value.  
The final metric flags generated values that are **outside** the examples.


In [ ]:
# Optional install cell
# Run this once if the libraries are not installed in your environment.
# DataSynthesizer can be sensitive to Python/library versions; if it fails, keep Faker and SDV tests running.

%pip install -q pandas numpy openpyxl faker sdv sdmetrics DataSynthesizer


## 1. Configuration and input loading

This top section now contains **all input schema handling and generation-rule detection**.

Edit only this section when your Excel/CSV headers or special column rules change:

- `INPUT_FILE`
- `COLUMN_NAME_FIELD`
- `EXAMPLES_FIELD`
- optional `GENERATION_TYPE_FIELD`
- optional `DATE_FORMAT_FIELD`

Supported `generation_type` values:

| generation_type | What it does | Example |
|---|---|---|
| `categorical` | Generates only from example values | `Delhi, Mumbai, Pune` |
| `date` | Generates random dates in same format/range | `01/01/2024, 12/31/2024` or `MM/DD/YYYY` |
| `masked` | Generates anonymized values matching same pattern | `123XX45` -> `908XX21` |

If `generation_type` is not provided, the notebook will try to auto-detect date and masked columns from the example values.


In [ ]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================================================
# USER CONFIGURATION - EDIT THIS SECTION FIRST
# =========================================================

# Input file can be CSV or Excel.
# Expected structure by default:
# column_name | examples | generation_type(optional) | date_format(optional)
INPUT_FILE = "input_schema.csv"   # example: "input_schema.xlsx"

OUTPUT_DIR = Path("synthetic_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

N_ROWS = 500
RANDOM_SEED = 42

# Required headers in your Excel/CSV.
# If your file uses different names, change only these two values.
# Example: if Excel has "Column" and "Example Values", set:
# COLUMN_NAME_FIELD = "Column"
# EXAMPLES_FIELD = "Example Values"
COLUMN_NAME_FIELD = "column_name"
EXAMPLES_FIELD = "examples"

# Optional headers.
# If present in Excel/CSV, these override auto-detection.
# generation_type accepted values:
# - categorical
# - date
# - masked
GENERATION_TYPE_FIELD = "generation_type"
DATE_FORMAT_FIELD = "date_format"

# Default date behavior:
# - If actual example dates are provided, random dates are generated between min(example_dates) and max(example_dates).
# - If only a placeholder like MM/DD/YYYY is provided, dates are generated between DEFAULT_DATE_START and DEFAULT_DATE_END.
DEFAULT_DATE_START = "2020-01-01"
DEFAULT_DATE_END = "2026-12-31"

# Supported date formats. Add more here if your Excel uses another date format.
DATE_FORMAT_MAP = {
    "MM/DD/YYYY": ("%m/%d/%Y", "MM/DD/YYYY"),
    "M/D/YYYY": ("%m/%d/%Y", "MM/DD/YYYY"),
    "YYYY-MM-DD": ("%Y-%m-%d", "YYYY-MM-DD"),
    "DD/MM/YYYY": ("%d/%m/%Y", "DD/MM/YYYY"),
    "DD-MM-YYYY": ("%d-%m-%Y", "DD-MM-YYYY"),
}

# Aliases accepted in generation_type column.
DATE_TYPE_ALIASES = {"date", "date_random", "random_date"}
MASKED_TYPE_ALIASES = {"masked", "mask", "anonymized", "anonymised", "hidden"}
CATEGORICAL_TYPE_ALIASES = {"categorical", "category", "exact", "sample"}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =========================================================
# INPUT LOADING + RULE DETECTION HELPERS
# All date/masked/categorical logic starts here.
# =========================================================

def split_examples(value: Any) -> List[str]:
    """
    Splits example values from a cell.
    Supports comma, pipe, semicolon, or newline separated values.
    Example: "Delhi, Mumbai, Pune" -> ["Delhi", "Mumbai", "Pune"]
    """
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[,|;\n]+", text)
    return [p.strip() for p in parts if p.strip() != ""]

def is_date_placeholder(value: str) -> bool:
    value = str(value).strip().upper()
    return value in DATE_FORMAT_MAP

def detect_date_format(values: List[str], explicit_format: Optional[str] = None) -> Optional[Tuple[str, str]]:
    """
    Returns (python_strftime_format, display_format) if a date format is detected.
    Main target: MM/DD/YYYY.
    """
    fmt = str(explicit_format).strip() if explicit_format is not None and not pd.isna(explicit_format) else ""
    fmt_upper = fmt.upper()

    if fmt_upper in DATE_FORMAT_MAP:
        return DATE_FORMAT_MAP[fmt_upper]

    clean = [str(v).strip() for v in values if str(v).strip()]

    # Case: example value itself is only a placeholder like MM/DD/YYYY.
    if clean and all(v.upper() in DATE_FORMAT_MAP for v in clean):
        return DATE_FORMAT_MAP[clean[0].upper()]

    # Detect actual values like 04/25/2026 as MM/DD/YYYY.
    actual = [v for v in clean if not is_date_placeholder(v)]
    if actual and all(re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", v) for v in actual):
        parsed = pd.to_datetime(actual, format="%m/%d/%Y", errors="coerce")
        if pd.Series(parsed).notna().all():
            return ("%m/%d/%Y", "MM/DD/YYYY")

    # Detect actual values like 2026-04-25 as YYYY-MM-DD.
    if actual and all(re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", v) for v in actual):
        parsed = pd.to_datetime(actual, format="%Y-%m-%d", errors="coerce")
        if pd.Series(parsed).notna().all():
            return ("%Y-%m-%d", "YYYY-MM-DD")

    return None

def detect_masked_pattern(values: List[str]) -> Optional[str]:
    """
    Detects masked/anonymized examples like 123XX45.
    X positions are preserved as X; digit positions are randomized.
    """
    clean = [str(v).strip() for v in values if str(v).strip()]
    if not clean:
        return None

    # Use the first value with X/x as the pattern.
    # Example: 123XX45 -> random digits in digit places, X remains X.
    for v in clean:
        if "X" in v.upper() and re.fullmatch(r"[A-Za-z0-9_\-./]+", v):
            return v
    return None

def infer_series_type(values: List[str]) -> str:
    """
    Normal exact-domain type inference.
    Note: dates and masked columns are handled by column rules first.
    """
    if not values:
        return "categorical"

    try:
        [int(v) for v in values]
        return "integer"
    except Exception:
        pass

    try:
        [float(v) for v in values]
        return "float"
    except Exception:
        pass

    return "categorical"

def cast_value(value: str, dtype: str):
    if dtype == "integer":
        return int(float(value))
    if dtype == "float":
        return float(value)
    return str(value)

def parse_actual_dates(values: List[str], py_fmt: str) -> List[pd.Timestamp]:
    dates = []
    for v in values:
        v = str(v).strip()
        if not v or is_date_placeholder(v):
            continue
        parsed = pd.to_datetime(v, format=py_fmt, errors="coerce")
        if not pd.isna(parsed):
            dates.append(pd.Timestamp(parsed))
    return dates

def random_date_string(py_fmt: str, values: List[str]) -> str:
    actual_dates = parse_actual_dates(values, py_fmt)
    if len(actual_dates) >= 2:
        start = min(actual_dates)
        end = max(actual_dates)
    else:
        start = pd.Timestamp(DEFAULT_DATE_START)
        end = pd.Timestamp(DEFAULT_DATE_END)

    if end < start:
        start, end = end, start

    days = max((end - start).days, 0)
    offset = random.randint(0, days) if days > 0 else 0
    return (start + pd.Timedelta(days=offset)).strftime(py_fmt)

def random_masked_value(pattern: str) -> str:
    """
    Example:
    pattern 123XX45 can generate 908XX21.
    pattern AB12XX can generate QZ87XX.
    """
    out = []
    for ch in str(pattern):
        if ch.upper() == "X":
            out.append("X")
        elif ch.isdigit():
            out.append(str(random.randint(0, 9)))
        elif ch.isalpha():
            new_ch = chr(random.randint(ord("A"), ord("Z")))
            out.append(new_ch if ch.isupper() else new_ch.lower())
        else:
            out.append(ch)
    return "".join(out)

def load_schema_examples(path: str):
    """
    Loads the Excel/CSV and creates three things used by all later cells:

    1. allowed_raw:
       Original allowed example values per column.

    2. inferred_types:
       Type or rule per column.
       Possible values: categorical, integer, float, date_random, masked_random

    3. column_rules:
       Validation/generation rules used by Faker, SDV, DataSynthesizer, and metrics.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Create input_schema.csv or input_schema.xlsx with columns: "
            f"{COLUMN_NAME_FIELD}, {EXAMPLES_FIELD}"
        )

    if path.suffix.lower() in [".xlsx", ".xls"]:
        schema_df = pd.read_excel(path)
    else:
        schema_df = pd.read_csv(path)

    required = {COLUMN_NAME_FIELD, EXAMPLES_FIELD}
    missing = required - set(schema_df.columns)
    if missing:
        raise ValueError(f"Input file missing columns: {missing}. Required columns are: {required}")

    allowed_raw = {}
    inferred_types = {}
    column_rules = {}

    for _, row in schema_df.iterrows():
        col = str(row[COLUMN_NAME_FIELD]).strip()
        examples = split_examples(row[EXAMPLES_FIELD])
        if not col or not examples:
            continue

        explicit_generation_type = ""
        if GENERATION_TYPE_FIELD in schema_df.columns and not pd.isna(row.get(GENERATION_TYPE_FIELD)):
            explicit_generation_type = str(row.get(GENERATION_TYPE_FIELD)).strip().lower()

        explicit_date_format = row.get(DATE_FORMAT_FIELD) if DATE_FORMAT_FIELD in schema_df.columns else None

        allowed_raw[col] = examples

        # Detection happens here in the input loading cell.
        date_format_info = detect_date_format(examples, explicit_date_format)
        masked_pattern = detect_masked_pattern(examples)

        if explicit_generation_type in DATE_TYPE_ALIASES or date_format_info:
            py_fmt, display_fmt = date_format_info or ("%m/%d/%Y", "MM/DD/YYYY")
            inferred_types[col] = "date_random"
            column_rules[col] = {
                "generation_type": "date_random",
                "python_date_format": py_fmt,
                "display_format": display_fmt,
                "validation": "date_format_and_optional_range",
                "rule_source": "explicit_generation_type" if explicit_generation_type in DATE_TYPE_ALIASES else "auto_detected_from_examples"
            }
        elif explicit_generation_type in MASKED_TYPE_ALIASES or masked_pattern:
            pattern = masked_pattern or examples[0]
            inferred_types[col] = "masked_random"
            column_rules[col] = {
                "generation_type": "masked_random",
                "pattern": pattern,
                "validation": "masked_pattern_match",
                "rule_source": "explicit_generation_type" if explicit_generation_type in MASKED_TYPE_ALIASES else "auto_detected_from_examples"
            }
        else:
            dtype = infer_series_type(examples)
            inferred_types[col] = dtype
            column_rules[col] = {
                "generation_type": "categorical_exact",
                "validation": "must_be_one_of_examples",
                "rule_source": "default_exact_examples"
            }

    if not allowed_raw:
        raise ValueError("No valid columns/examples found in input file.")

    return allowed_raw, inferred_types, column_rules, schema_df

def generate_value_for_column(col: str):
    rule = column_rules[col]
    values = allowed_raw[col]
    dtype = inferred_types[col]

    if rule["generation_type"] == "date_random":
        return random_date_string(rule["python_date_format"], values)

    if rule["generation_type"] == "masked_random":
        return random_masked_value(rule["pattern"])

    chosen = random.choice(values)
    return cast_value(chosen, dtype)

def post_process_special_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Forces date_random and masked_random columns to follow their requested rule.
    This is useful for SDV/DataSynthesizer because they may otherwise only reproduce examples
    or create values that do not follow your anonymization/date format requirements.
    """
    df = df.copy()
    for col, rule in column_rules.items():
        if col not in df.columns:
            continue
        if rule["generation_type"] in {"date_random", "masked_random"}:
            df[col] = [generate_value_for_column(col) for _ in range(len(df))]
    return df

def build_seed_dataframe(allowed_raw, inferred_types, n_rows: int = 200) -> pd.DataFrame:
    """
    Builds a small 'real-like' training dataset from the examples/rules.
    - categorical_exact: samples only from examples
    - date_random: generates random dates in the requested format/range
    - masked_random: generates anonymized values matching the pattern
    """
    rows = []
    for _ in range(n_rows):
        rows.append({col: generate_value_for_column(col) for col in allowed_raw.keys()})
    return pd.DataFrame(rows)

# Load once here. Every later library cell uses these same rules.
allowed_raw, inferred_types, column_rules, schema_df = load_schema_examples(INPUT_FILE)
real_seed_df = build_seed_dataframe(allowed_raw, inferred_types, n_rows=max(200, N_ROWS))

print("Loaded schema:")
display(schema_df)

print("Detected generation rules from the top input-loading cell:")
display(pd.DataFrame([
    {"column_name": col, "inferred_type": inferred_types[col], **column_rules[col]}
    for col in allowed_raw.keys()
]))

print("Seed training data sample used by SDV and DataSynthesizer:")
display(real_seed_df.head())


## 2. Create a sample input file if needed


In [ ]:
# Run this cell only if you want a quick sample schema file.
# It shows all supported cases:
# - exact categorical values
# - random date generation
# - masked/anonymized value generation

sample_schema = pd.DataFrame({
    "column_name": [
        "customer_name",
        "city",
        "status",
        "amount",
        "transaction_date",
        "statement_date_range",
        "masked_account_id"
    ],
    "examples": [
        "Shivam, Rahul, Aman, Priya",
        "Delhi, Mumbai, Pune, Bengaluru",
        "ACTIVE, INACTIVE, BLOCKED",
        "100, 200, 500, 1000",
        "MM/DD/YYYY",
        "01/01/2024, 12/31/2024",
        "123XX45"
    ],
    "generation_type": [
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "date",
        "date",
        "masked"
    ],
    "date_format": [
        "",
        "",
        "",
        "",
        "MM/DD/YYYY",
        "MM/DD/YYYY",
        ""
    ]
})

sample_schema.to_csv("input_schema_sample.csv", index=False)
sample_schema.to_excel("input_schema_sample.xlsx", index=False)

print("Saved input_schema_sample.csv and input_schema_sample.xlsx")
display(sample_schema)


## 3. Faker generation

Faker usually generates realistic-looking values using providers like name, email, address, date, etc.  
For this test, because your requirement is **only from provided example values**, this cell uses Faker's random engine to sample from the allowed examples.


In [ ]:
from faker import Faker

fake = Faker()
Faker.seed(RANDOM_SEED)

faker_rows = []
for _ in range(N_ROWS):
    row = {}
    for col in allowed_raw.keys():
        row[col] = generate_value_for_column(col)
    faker_rows.append(row)

faker_df = pd.DataFrame(faker_rows)

faker_csv = OUTPUT_DIR / "faker_synthetic.csv"
faker_xlsx = OUTPUT_DIR / "faker_synthetic.xlsx"

faker_df.to_csv(faker_csv, index=False)
faker_df.to_excel(faker_xlsx, index=False)

print(f"Saved: {faker_csv}")
print(f"Saved: {faker_xlsx}")
display(faker_df.head())


## 4. SDV generation

SDV learns a statistical model from tabular data and samples new rows from that model.  
Here, the seed data is created only from your example values. SDV may still generate values outside the original examples, especially for numeric columns, so the metric section checks that.

In this notebook numeric columns are deliberately marked as categorical because your stated requirement is exact allowed-value generation from the examples.


In [ ]:
try:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import GaussianCopulaSynthesizer

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(real_seed_df)

    # Treat all columns as categorical because your main requirement is controlled generation.
    # Then we post-process date/masked columns to force date format and anonymized patterns.
    for col in real_seed_df.columns:
        metadata.update_column(column_name=col, sdtype="categorical")

    sdv_model = GaussianCopulaSynthesizer(metadata)
    sdv_model.fit(real_seed_df)
    sdv_df = sdv_model.sample(num_rows=N_ROWS)

    # Force special columns like date_random and masked_random to follow requested rules.
    sdv_df = post_process_special_columns(sdv_df)

    # Normalize regular numeric columns where applicable.
    for col, dtype in inferred_types.items():
        if col in sdv_df.columns:
            if dtype == "integer":
                sdv_df[col] = pd.to_numeric(sdv_df[col], errors="coerce").round().astype("Int64")
            elif dtype == "float":
                sdv_df[col] = pd.to_numeric(sdv_df[col], errors="coerce")

    sdv_csv = OUTPUT_DIR / "sdv_synthetic.csv"
    sdv_xlsx = OUTPUT_DIR / "sdv_synthetic.xlsx"

    sdv_df.to_csv(sdv_csv, index=False)
    sdv_df.to_excel(sdv_xlsx, index=False)

    print(f"Saved: {sdv_csv}")
    print(f"Saved: {sdv_xlsx}")
    display(sdv_df.head())

except Exception as e:
    print("SDV generation failed.")
    print("Error:", repr(e))
    sdv_df = pd.DataFrame()


## 5. DataSynthesizer generation

DataSynthesizer learns distributions from a dataset and can generate synthetic records from those learned distributions.  
It is older than Faker/SDV and may be more sensitive to environment versions. This cell uses the independent attribute mode.


In [ ]:
try:
    from DataSynthesizer.DataDescriber import DataDescriber
    from DataSynthesizer.DataGenerator import DataGenerator

    ds_input = OUTPUT_DIR / "datasynthesizer_seed.csv"
    description_file = OUTPUT_DIR / "datasynthesizer_description.json"
    ds_output = OUTPUT_DIR / "datasynthesizer_synthetic.csv"
    ds_xlsx = OUTPUT_DIR / "datasynthesizer_synthetic.xlsx"

    # DataSynthesizer works best with CSV input.
    ds_seed_df = real_seed_df.copy()
    ds_seed_df.to_csv(ds_input, index=False)

    categorical_map = {col: True for col in ds_seed_df.columns}
    candidate_key_map = {col: False for col in ds_seed_df.columns}

    describer = DataDescriber(category_threshold=1000)

    # DataSynthesizer versions differ slightly. Some accept epsilon, some do not.
    try:
        describer.describe_dataset_in_independent_attribute_mode(
            dataset_file=str(ds_input),
            epsilon=0,
            attribute_to_is_categorical=categorical_map,
            attribute_to_is_candidate_key=candidate_key_map
        )
    except TypeError:
        describer.describe_dataset_in_independent_attribute_mode(
            dataset_file=str(ds_input),
            attribute_to_is_categorical=categorical_map,
            attribute_to_is_candidate_key=candidate_key_map
        )

    describer.save_dataset_description_to_file(str(description_file))

    generator = DataGenerator()
    generator.generate_dataset_in_independent_mode(N_ROWS, str(description_file))
    generator.save_synthetic_data(str(ds_output))

    datasynth_df = pd.read_csv(ds_output)

    # Force special columns like date_random and masked_random to follow requested rules.
    datasynth_df = post_process_special_columns(datasynth_df)

    # Normalize regular numeric columns where applicable.
    for col, dtype in inferred_types.items():
        if col in datasynth_df.columns:
            if dtype == "integer":
                datasynth_df[col] = pd.to_numeric(datasynth_df[col], errors="coerce").round().astype("Int64")
            elif dtype == "float":
                datasynth_df[col] = pd.to_numeric(datasynth_df[col], errors="coerce")

    datasynth_df.to_csv(ds_output, index=False)
    datasynth_df.to_excel(ds_xlsx, index=False)

    print(f"Saved: {ds_output}")
    print(f"Saved: {ds_xlsx}")
    display(datasynth_df.head())

except Exception as e:
    print("DataSynthesizer generation failed.")
    print("Error:", repr(e))
    datasynth_df = pd.DataFrame()


## 6. Metric: check if generated data stayed within examples

This metric can run against each generated CSV/XLSX independently.  
It answers:

1. Did the output contain expected columns?
2. Did it create any extra columns?
3. For each column, how many generated values are outside the allowed examples?
4. Which invalid values appeared?
5. What percentage of rows are valid?


In [ ]:
def normalize_for_compare(value: Any, dtype: str) -> str:
    if pd.isna(value):
        return "<NULL>"
    if dtype == "integer":
        try:
            return str(int(float(value)))
        except Exception:
            return str(value).strip()
    if dtype == "float":
        try:
            f = float(value)
            return str(int(f)) if f.is_integer() else str(f)
        except Exception:
            return str(value).strip()
    return str(value).strip()

def read_generated_file(path):
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)

def masked_pattern_to_regex(pattern: str) -> str:
    regex_parts = []
    for ch in str(pattern):
        if ch.upper() == "X":
            regex_parts.append("X")
        elif ch.isdigit():
            regex_parts.append(r"\d")
        elif ch.isalpha():
            regex_parts.append(r"[A-Za-z]")
        else:
            regex_parts.append(re.escape(ch))
    return "^" + "".join(regex_parts) + "$"

def validate_date_value(value: Any, col: str) -> Tuple[bool, str]:
    if pd.isna(value):
        return False, "NULL_DATE"

    value_str = str(value).strip()
    py_fmt = column_rules[col]["python_date_format"]
    display_fmt = column_rules[col]["display_format"]

    parsed = pd.to_datetime(value_str, format=py_fmt, errors="coerce")
    if pd.isna(parsed):
        return False, f"INVALID_DATE_FORMAT_EXPECTED_{display_fmt}"

    # If actual example dates were given, also validate generated dates are within example min/max.
    actual_dates = parse_actual_dates(allowed_raw[col], py_fmt)
    if len(actual_dates) >= 2:
        start = min(actual_dates)
        end = max(actual_dates)
        if not (start <= pd.Timestamp(parsed) <= end):
            return False, f"DATE_OUTSIDE_EXAMPLE_RANGE_{start.strftime(py_fmt)}_TO_{end.strftime(py_fmt)}"

    return True, "PASS"

def validate_masked_value(value: Any, col: str) -> Tuple[bool, str]:
    if pd.isna(value):
        return False, "NULL_MASKED_VALUE"
    pattern = column_rules[col]["pattern"]
    regex = masked_pattern_to_regex(pattern)
    value_str = str(value).strip()
    if re.fullmatch(regex, value_str):
        return True, "PASS"
    return False, f"PATTERN_MISMATCH_EXPECTED_{pattern}"

def validate_generated_file(generated_file, allowed_raw, inferred_types, save_report=True):
    generated_file = Path(generated_file)
    df = read_generated_file(generated_file)

    expected_cols = set(allowed_raw.keys())
    actual_cols = set(df.columns)

    missing_cols = sorted(expected_cols - actual_cols)
    extra_cols = sorted(actual_cols - expected_cols)

    summary_rows = []
    invalid_detail_rows = []

    for col in sorted(expected_cols):
        dtype = inferred_types[col]
        rule = column_rules[col]

        if col not in df.columns:
            summary_rows.append({
                "file": generated_file.name,
                "column": col,
                "generation_type": rule["generation_type"],
                "status": "MISSING_COLUMN",
                "rows": len(df),
                "valid_count": 0,
                "invalid_count": len(df),
                "null_count": None,
                "valid_pct": 0.0,
                "validation_rule": rule["validation"],
                "invalid_values_or_reasons": "COLUMN_NOT_FOUND"
            })
            continue

        valid_flags = []
        reasons = []
        normalized_values = []

        if rule["generation_type"] == "date_random":
            for x in df[col]:
                valid, reason = validate_date_value(x, col)
                valid_flags.append(valid)
                reasons.append(reason)
                normalized_values.append(str(x).strip() if not pd.isna(x) else "<NULL>")

        elif rule["generation_type"] == "masked_random":
            for x in df[col]:
                valid, reason = validate_masked_value(x, col)
                valid_flags.append(valid)
                reasons.append(reason)
                normalized_values.append(str(x).strip() if not pd.isna(x) else "<NULL>")

        else:
            allowed_norm = {normalize_for_compare(v, dtype) for v in allowed_raw[col]}
            for x in df[col]:
                x_norm = normalize_for_compare(x, dtype)
                valid = x_norm in allowed_norm
                valid_flags.append(valid)
                reasons.append("PASS" if valid else "VALUE_NOT_IN_EXAMPLES")
                normalized_values.append(x_norm)

        valid_series = pd.Series(valid_flags)
        invalid_mask = ~valid_series
        null_count = int(pd.Series(normalized_values).eq("<NULL>").sum())

        invalid_values = sorted(set(
            f"{normalized_values[i]} => {reasons[i]}"
            for i, is_invalid in enumerate(invalid_mask)
            if is_invalid
        ))

        summary_rows.append({
            "file": generated_file.name,
            "column": col,
            "generation_type": rule["generation_type"],
            "status": "PASS" if invalid_mask.sum() == 0 else "FAIL",
            "rows": len(df),
            "valid_count": int(valid_series.sum()),
            "invalid_count": int(invalid_mask.sum()),
            "null_count": null_count,
            "valid_pct": round(valid_series.mean() * 100, 2) if len(df) else 0.0,
            "validation_rule": rule["validation"],
            "invalid_values_or_reasons": ", ".join(map(str, invalid_values[:20]))
        })

        if invalid_mask.sum() > 0:
            for i, is_invalid in enumerate(invalid_mask):
                if is_invalid:
                    invalid_detail_rows.append({
                        "file": generated_file.name,
                        "column": col,
                        "generation_type": rule["generation_type"],
                        "generated_value": normalized_values[i],
                        "reason": reasons[i]
                    })

    if missing_cols or extra_cols:
        summary_rows.insert(0, {
            "file": generated_file.name,
            "column": "__FILE_STRUCTURE__",
            "generation_type": "structure",
            "status": "FAIL",
            "rows": len(df),
            "valid_count": None,
            "invalid_count": None,
            "null_count": None,
            "valid_pct": None,
            "validation_rule": "expected_columns_match_generated_columns",
            "invalid_values_or_reasons": f"missing_cols={missing_cols}; extra_cols={extra_cols}"
        })

    summary = pd.DataFrame(summary_rows)
    invalid_details = pd.DataFrame(invalid_detail_rows)

    if save_report:
        report_base = OUTPUT_DIR / f"metric_report_{generated_file.stem}"
        summary.to_csv(f"{report_base}_summary.csv", index=False)
        summary.to_excel(f"{report_base}_summary.xlsx", index=False)
        if not invalid_details.empty:
            invalid_details.to_csv(f"{report_base}_invalid_values.csv", index=False)
            invalid_details.to_excel(f"{report_base}_invalid_values.xlsx", index=False)

    return summary, invalid_details

# Run metrics individually on any generated output file.
# You can comment/uncomment whichever file you want to test.
generated_files_to_check = [
    OUTPUT_DIR / "faker_synthetic.csv",
    OUTPUT_DIR / "sdv_synthetic.csv",
    OUTPUT_DIR / "datasynthesizer_synthetic.csv"
]

all_summaries = []
all_invalid_details = []

for file in generated_files_to_check:
    if Path(file).exists():
        print(f"\nValidating: {file}")
        summary, details = validate_generated_file(file, allowed_raw, inferred_types)
        display(summary)
        all_summaries.append(summary)
        if not details.empty:
            all_invalid_details.append(details)
    else:
        print(f"Skipping missing file: {file}")

if all_summaries:
    combined_summary = pd.concat(all_summaries, ignore_index=True)
    combined_summary.to_csv(OUTPUT_DIR / "metric_report_combined_summary.csv", index=False)
    combined_summary.to_excel(OUTPUT_DIR / "metric_report_combined_summary.xlsx", index=False)
    print("\nCombined summary saved.")
    display(combined_summary)

if all_invalid_details:
    combined_invalid_details = pd.concat(all_invalid_details, ignore_index=True)
    combined_invalid_details.to_csv(OUTPUT_DIR / "metric_report_combined_invalid_values.csv", index=False)
    combined_invalid_details.to_excel(OUTPUT_DIR / "metric_report_combined_invalid_values.xlsx", index=False)
    print("Combined invalid details saved.")
    display(combined_invalid_details.head(50))
else:
    print("No invalid values found in checked files.")


## 7. Optional: compare statistical similarity using SDMetrics


In [ ]:
# This section is optional.
# It compares generated data against the seed training data statistically.
# The domain validation above is still the main metric for your exact requirement.

try:
    from sdmetrics.reports.single_table import QualityReport
    from sdv.metadata import SingleTableMetadata

    def run_sdmetrics_quality_report(synthetic_df: pd.DataFrame, name: str):
        if synthetic_df is None or synthetic_df.empty:
            print(f"Skipping {name}: empty data")
            return None

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(real_seed_df)

        report = QualityReport()
        report.generate(real_seed_df, synthetic_df[real_seed_df.columns], metadata.to_dict())

        print(f"{name} quality score:", report.get_score())
        details = report.get_details(property_name="Column Shapes")
        display(details)
        return report

    if "faker_df" in globals():
        faker_quality = run_sdmetrics_quality_report(faker_df, "Faker")
    if "sdv_df" in globals() and not sdv_df.empty:
        sdv_quality = run_sdmetrics_quality_report(sdv_df, "SDV")
    if "datasynth_df" in globals() and not datasynth_df.empty:
        datasynth_quality = run_sdmetrics_quality_report(datasynth_df, "DataSynthesizer")

except Exception as e:
    print("Optional SDMetrics report failed.")
    print("Error:", repr(e))


## Output files

After running the notebook, check the `synthetic_outputs/` folder.

Expected generated files:

- `faker_synthetic.csv`
- `sdv_synthetic.csv`
- `datasynthesizer_synthetic.csv`

Expected metric files:

- `metric_report_faker_synthetic_summary.csv`
- `metric_report_sdv_synthetic_summary.csv`
- `metric_report_datasynthesizer_synthetic_summary.csv`
- `combined_metric_summary.csv`
